In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sharedfunctions import (loadKernels, computeSigma, CSVPATH, IDXROBUST, IDXVULNERABLE)

In [2]:
def generateVPWaveletKernel():
    n=3
    nodes = np.cos(np.pi * np.arange(1, 2*n, 2) / (2*n))
    vp2d = np.outer(nodes, nodes)
    vp2d = vp2d - vp2d.mean()
    vp2d = vp2d / np.max(np.abs(vp2d))
    return vp2d.astype(np.float32)


In [3]:
kernels = loadKernels(CSVPATH)

In [4]:


print("\n" + "="*60)
print("Vulnerability Threshold Justification")
print("="*60)

vp = generateVPWaveletKernel()

sigmas = np.array([computeSigma(h) for h in kernels])
meanSigma = np.mean(sigmas)

sigmaVp = np.min(np.abs(np.fft.fft2(vp, s=(256, 256))))
n = 3
nodes = np.cos(np.pi * np.arange(1, 2*n, 2) / (2*n))
vpRaw = np.outer(nodes, nodes)
vpRaw = (vpRaw / np.max(np.abs(vpRaw))).astype(np.float32)
sigmaVp = np.min(np.abs(np.fft.fft2(vpRaw, s=(256, 256))))
sigmaRobustAnchor = sigmas[IDXROBUST]


print(f"\n  VP Wavelet (Gold Standard) sigma = {sigmaVp:.6f}")
print(f"  This is the empirical anchor for the Robust threshold.\n")




thresholds = {
        "Critical":  (0.000, 0.001),
        "High Risk": (0.001, 0.005),
        "Moderate":  (0.005, 0.050),
        "Robust":    (0.050, np.inf),
    }

print("  Threshold Band Justification Table")
print("  " + "-"*70)
print(f"  {'Band':<12} {'Range':<20} {'Count':>6} {'%':>6}  {'Rationale'}")
print("  " + "-"*70)

rationales = {
        "Critical":  "sigma < epsilon_min; fails under infinitesimal perturbation",
        "High Risk": "fails within standard FGSM threat model (eps=0.001-0.005)",
        "Moderate":  "vulnerable to domain shift (e.g. California -> Norway)",
        "Robust":    "within VP Wavelet order-of-magnitude; spectrally safe",
    }

for band, (lo, hi) in thresholds.items():
        if hi == np.inf:
            mask = sigmas >= lo
            rangeStr = f">= {lo:.3f}"
        else:
            mask = (sigmas >= lo) & (sigmas < hi)
            rangeStr = f"[{lo:.3f}, {hi:.3f})"
        count = mask.sum()
        pct = 100 * count / len(sigmas)
        print(f"  {band:<12} {rangeStr:<20} {count:>6} {pct:>5.1f}%  {rationales[band]}")

print("  " + "-"*70)

print(f"\n  VP Wavelet sigma = {sigmaVp:.6f}  →  falls in: ", end="")

if sigmaVp >= 0.05:
        print("Robust band ✓ (confirms threshold calibration)")
elif sigmaVp >= 0.005:
        print("Moderate band — consider adjusting Robust threshold")
else:
        print("High Risk band — VP wavelet construction should be verified")

print(f"\n  Kernel {IDXROBUST} sigma = {sigmas[IDXROBUST]:.6f}  "
          f"(ratio to VP: {sigmas[IDXROBUST]/sigmaVp:.2f}x)")
print(f"  Kernel {IDXVULNERABLE} sigma = {sigmas[IDXVULNERABLE]:.8f}  "
          f"(ratio to VP: {sigmas[IDXVULNERABLE]/(sigmaVp+1e-12):.4f}x)")


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Vulnerability Threshold Justification\nKernel 42 Sigma Anchors the Robust Threshold Empirically",
    fontsize=12, fontweight='bold'
)


ax = axes[0]
colorsBand = {
        "Critical":  ("#ffcccc", 0.000, 0.001),
        "High Risk": ("#ffe0b3", 0.001, 0.005),
        "Moderate":  ("#fffacc", 0.005, 0.050),
        "Robust":    ("#ccffcc", 0.050, max(sigmas.max(), sigmaVp) * 1.1),
    }
for label, (color, lo, hi) in colorsBand.items():
        ax.axvspan(lo, hi, alpha=0.35, color=color, label=label)

ax.hist(sigmas, bins=60, color='#5c6bc0', edgecolor='white', alpha=0.85, zorder=3)
ax.axvline(meanSigma, color='red', linewidth=2, linestyle='--',
               label=f'Mean sigma: {meanSigma:.4f}', zorder=4)
ax.axvline(sigmas[IDXROBUST], color='orange', linewidth=2, linestyle='-.',
               label=f'Kernel {IDXROBUST} (Robust): {sigmas[IDXROBUST]:.4f}', zorder=4)
ax.axvline(sigmas[IDXVULNERABLE], color='blue', linewidth=2, linestyle=':',
               label=f'Kernel {IDXVULNERABLE} (Vulnerable): {sigmas[IDXVULNERABLE]:.2e}', zorder=4)


ax.set_title("Sigma Distribution with Kernel 42 Robust Anchor\n"
             "Green border = empirically calibrated Robust band",
             fontsize=10, fontweight='bold')
ax.set_xlabel("Stability Margin (sigma)", fontsize=10)
ax.set_ylabel("Kernel Count", fontsize=10)
ax.set_xlim(-0.005, min(sigmas.max() * 1.05, 0.5))
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3, zorder=0)

 
ax2 = axes[1]
bandNames = list(thresholds.keys())
bandColors = ['#ff6b6b', '#ffa94d', '#ffe066', '#69db7c']
bandCounts = []
for band, (lo, hi) in thresholds.items():
        if hi == np.inf:
            bandCounts.append(100 * (sigmas >= lo).mean())
        else:
            bandCounts.append(100 * ((sigmas >= lo) & (sigmas < hi)).mean())

bars = ax2.bar(bandNames, bandCounts, color=bandColors,
                   edgecolor='white', linewidth=1.2)
for bar, pct in zip(bars, bandCounts):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')



ax2.set_ylabel("% of All Kernels", fontsize=10)
ax2.set_ylim(0, max(bandCounts) * 1.2)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("ThresholdJustification.png", dpi=150, bbox_inches='tight')
plt.close()



    




Vulnerability Threshold Justification

  VP Wavelet (Gold Standard) sigma = 0.000000
  This is the empirical anchor for the Robust threshold.

  Threshold Band Justification Table
  ----------------------------------------------------------------------
  Band         Range                 Count      %  Rationale
  ----------------------------------------------------------------------
  Critical     [0.000, 0.001)          929  22.7%  sigma < epsilon_min; fails under infinitesimal perturbation
  High Risk    [0.001, 0.005)         1918  46.8%  fails within standard FGSM threat model (eps=0.001-0.005)
  Moderate     [0.005, 0.050)         1159  28.3%  vulnerable to domain shift (e.g. California -> Norway)
  Robust       >= 0.050                 90   2.2%  within VP Wavelet order-of-magnitude; spectrally safe
  ----------------------------------------------------------------------

  VP Wavelet sigma = 0.000000  →  falls in: High Risk band — VP wavelet construction should be verified

  